In [ ]:
import cv2, numpy as np

rng = np.random.default_rng(0)
base = rng.integers(60, 200, (700, 900), dtype=np.uint8)
base = cv2.GaussianBlur(base, (0, 0), 3)
base = cv2.normalize(base, None, 0, 255, cv2.NORM_MINMAX)
for _ in range(250):  # "pedras"
    x, y, r = rng.integers(50, 850), rng.integers(50, 650), rng.integers(4, 12)
    cv2.circle(base, (int(x), int(y)), int(r), int(rng.integers(0, 255)), -1)

def frame(dx, dy):
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    return cv2.warpAffine(base, M, (900, 700), borderMode=cv2.BORDER_REFLECT)

# um frame a cada 10 min; a terra desce 0.8 px por frame (4.8 px/h)
frames = [frame(0, 0.8 * i) for i in range(7)]
tempos_h = [i * 10 / 60 for i in range(7)]

In [ ]:
def medir_movimento(f0, f1, dt_h):
    p0 = cv2.goodFeaturesToTrack(f0, maxCorners=300, qualityLevel=0.01, minDistance=10)
    p1, st, err = cv2.calcOpticalFlowPyrLK(f0, f1, p0, None, winSize=(21, 21), maxLevel=3)
    ok = st.flatten() == 1
    d = (p1[ok] - p0[ok]).reshape(-1, 2)
    dx, dy = np.median(d[:, 0]), np.median(d[:, 1])  # mediana ignora pontos ruins
    return dx / dt_h, dy / dt_h, int(ok.sum())

vx, vy, n = medir_movimento(frames[0], frames[-1], tempos_h[-1] - tempos_h[0])
print(f"pontos rastreados: {n}")
print(f"velocidade: vx={vx:.2f} px/h, vy={vy:.2f} px/h")

# aceleração: velocidade em duas janelas
v1 = medir_movimento(frames[0], frames[3], tempos_h[3])[1]
v2 = medir_movimento(frames[3], frames[6], tempos_h[6] - tempos_h[3])[1]
print(f"vy 1ª metade={v1:.2f}, 2ª metade={v2:.2f}, aceleração={(v2-v1)/(tempos_h[6]/2):.2f} px/h²")